###Deltalake & Lakehouse Optimization Usecases

![](/Workspace/Users/infoblisstech@gmail.com/databricks-code-repo/5_all_databricks_workouts/DELTA OPTIMIZATIONS.png)

####1. Handling Data Skew & Query Performance (Optimize & Z-Order)
Scenario: The analytics team reports that queries filtering silver_shipments by source_city and shipment_date are becoming slow as data volume grows.

Task: Run the OPTIMIZE command with ZORDER on the silver_shipments table to co-locate related data in the same files.

Outcome:
Why did we choose source_city and shipment_date for Z-Ordering instead of shipment_id? Think about high cardinality vs. query filtering

In [0]:
%sql
Create catalog If not exists Shipping_catalog_assign;
Create Schema If not exists Shipping_catalog_assign.Shipping_Schema;
Create volume If not exists Shipping_catalog_assign.Shipping_Schema.Shipping_vol;
  

In [0]:
%sql
USE CATALOG Shipping_catalog_assign;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS Shipping_catalog_assign.shipping_schema.silver_shipments (
  shipment_id STRING,
  source_city STRING,
  shipment_date DATE,
  weight_kg DOUBLE,
  status STRING
) USING DELTA;

In [0]:
%sql
-- Insert sample data with repeated cities and dates for clustering
INSERT INTO Shipping_catalog_assign.shipping_schema.silver_shipments VALUES
  ('SHP001', 'Seattle', '2024-01-01', 12.5, 'Delivered'),
  ('SHP002', 'Seattle', '2024-01-01', 8.2, 'In Transit'),
  ('SHP003', 'Chicago', '2024-01-01', 45.0, 'Delivered'),
  ('SHP004', 'Chicago', '2024-01-02', 2.1, 'Shipped'),
  ('SHP005', 'Seattle', '2024-01-02', 15.6, 'Delivered'),
  ('SHP006', 'New York', '2024-01-01', 30.0, 'In Transit'),
  ('SHP007', 'New York', '2024-01-02', 22.4, 'Delivered'),
  ('SHP008', 'Chicago', '2024-01-02', 10.0, 'Pending');

In [0]:
%sql
-- OPTIMIZE Shipping_catalog_assign.shipping_schema.silver_shipments 
-- ZORDER BY (source_city, shipment_date);
DESCRIBE HISTORY Shipping_catalog_assign.shipping_schema.silver_shipments;

#### 2. Speeding up Regional Queries (Partition Pruning)
Scenario: The dashboard team reports that queries filtering for orgin_hub_city with "New York" shipments from the gold_core_curated_tbl table are scanning the entire dataset (Terabytes of data), even though New York is only 5% of the data. This is racking up compute costs.

Task: Re-create the gold_core_curated_tbl table partitioned by orgin_hub_city. Run a query filtering for one city to demonstrate "Partition Pruning" (where Spark skips files that don't match the filter).

Outcome: Verify the partition filtering is applied or not, by performing explain plan, check for the PartitionFilters in the output.

In [0]:
%sql
-- Create the gold table partitioned by city
CREATE OR REPLACE TABLE Shipping_catalog_assign.shipping_schema.gold_core_curated_tbl (
  shipment_id STRING,
  destination_city STRING,
  shipment_date DATE,
  orgin_hub_city STRING -- This will be used for partitioning
) 
USING DELTA 
PARTITIONED BY (orgin_hub_city);

-- Insert sample data to demonstrate different partitions
INSERT INTO Shipping_catalog_assign.shipping_schema.gold_core_curated_tbl VALUES
  ('S001', 'Chicago', '2024-05-01', 'New York'),
  ('S002', 'Miami', '2024-05-02', 'New York'),
  ('S003', 'Seattle', '2024-05-01', 'Los Angeles'),
  ('S004', 'Boston', '2024-05-03', 'Chicago');

In [0]:
%sql
-- show partitions Shipping_catalog_assign.shipping_schema.gold_core_curated_tbl
-- DESCRIBE HISTORY Shipping_catalog_assign.shipping_schema.gold_core_curated_tbl;
SELECT * 
FROM Shipping_catalog_assign.shipping_schema.gold_core_curated_tbl 
WHERE orgin_hub_city = 'New York';

In [0]:
%sql
use Shipping_catalog_assign.shipping_schema

In [0]:
 %sql
CREATE OR REPLACE TABLE tblsales
(
  sales_id INT,
  product_id INT,
  region STRING,
  sales_amount DOUBLE,
  sales_date DATE
)
USING DELTA;

In [0]:
%sql
-- select * from tblsales
INSERT INTO tblsales VALUES
  (1, 101, 'North', 1000.50, '2025-10-16'),
  (2, 102, 'South', 500.75, '2025-10-16'),
  (3, 103, 'East', 700.20, '2025-10-16'),
  (4, 104, 'West', 1200.00, '2025-10-16');

INSERT INTO tblsales VALUES
  (5, 101, 'North', 800.00, '2025-10-17'),
  (6, 102, 'South', 450.00, '2025-10-17'),
  (7, 103, 'East', 600.00, '2025-10-17'),
  (8, 104, 'West', 1100.00, '2025-10-17');

In [0]:
%sql
-- DESCRIBE DETAIL tblsales;
-- OPTIMIZE tblsales;
-- DESCRIBE DETAIL tblsales;
-- CREATE OR REPLACE TABLE customer_txn (
--     txn_id INT,
--     customer_id INT,
--     region STRING,
--     txn_amount DOUBLE,
--     txn_type STRING,
--     transaction_date DATE
-- )
-- USING DELTA;
describe history customer_txn

In [0]:
%sql
--Step 2 – Insert multiple small batches
--Each insert writes a few small Parquet files.
-- Batch 1
INSERT INTO customer_txn VALUES
 (1, 1001, 'North', 250.00, 'Online', '2025-10-01'),
 (2, 1002, 'South', 400.00, 'Offline', '2025-10-02'),
 (3, 1003, 'West', 600.00, 'Online', '2025-10-03');

-- Batch 2
INSERT INTO customer_txn VALUES
 (4, 1001, 'North', 300.00, 'Offline', '2025-10-01'),
 (5, 1004, 'East', 750.00, 'Online', '2025-10-02'),
 (6, 1005, 'South', 180.00, 'Online', '2025-10-03');

-- Batch 3
INSERT INTO customer_txn VALUES
 (7, 1001, 'North', 270.00, 'Online', '2025-10-01'),
 (8, 1003, 'West', 500.00, 'Offline', '2025-10-02'),
 (9, 1002, 'South', 900.00, 'Online', '2025-10-03');

In [0]:
%sql
-- describe history customer_txn;
-- Step 3 – Inspect fragmentation (numFiles & sizeInBytes)
DESCRIBE DETAIL customer_txn;

In [0]:
%sql
CREATE OR REPLACE TABLE customer_txn (
    txn_id INT,
    customer_id INT,
    region STRING,
    txn_amount DOUBLE,
    txn_type STRING,
    transaction_date DATE
)
USING DELTA;

In [0]:
%sql
--Step 2 – Insert multiple small batches
--Each insert writes a few small Parquet files.
-- Batch 1
INSERT INTO customer_txn VALUES
 (1, 1001, 'North', 250.00, 'Online', '2025-10-01'),
 (2, 1002, 'South', 400.00, 'Offline', '2025-10-02'),
 (3, 1003, 'West', 600.00, 'Online', '2025-10-03');

-- Batch 2
INSERT INTO customer_txn VALUES
 (4, 1001, 'North', 300.00, 'Offline', '2025-10-01'),
 (5, 1004, 'East', 750.00, 'Online', '2025-10-02'),
 (6, 1005, 'South', 180.00, 'Online', '2025-10-03');

-- Batch 3
INSERT INTO customer_txn VALUES
 (7, 1001, 'North', 270.00, 'Online', '2025-10-01'),
 (8, 1003, 'West', 500.00, 'Offline', '2025-10-02'),
 (9, 1002, 'South', 900.00, 'Online', '2025-10-03');

In [0]:
%sql
CREATE OR REPLACE TABLE customer_txn_part1 (
    txn_id INT,
    customer_id INT,
    region STRING,
    txn_amount DOUBLE,
    txn_type STRING,
    transaction_date DATE
) 
using delta
partitioned by (transaction_date);
insert into customer_txn_part1 select * from Shipping_catalog_assign.shipping_schema.customer_txn;

In [0]:
%sql
-- show partitions customer_txn_part1
explain select * from customer_txn_part1 where transaction_date='2025-10-01'; --look at the 

#### 3. Storage Cost Savings (Vacuum)
Scenario: Your Project pipeline runs every hour, creating many small files and obsolete versions of data. Your storage costs are rising. You need to clean up files that are no longer needed for time travel.

Task: Execute a Vacuum command to remove data files older than the retention threshold.

Outcome: Performance improvement, cost saving, best practices.

Observation: Perform the describe history and find whether vacuum is completed.

####4. Modern Data Layout (Liquid Clustering)
Scenario: You are redesigning the silver_shipments table. You want to avoid the "small files" problem and need a flexible layout that adapts to changing query patterns automatically without rewriting the table.

Task: Re-create the silver_shipments table using Liquid Clustering on the shipment_id column.

Outcome: Liquid Clustering over traditional partitioning when the cardinality of shipment_id is very high.

#### 5. Cost Efficient Environment Cloning (Shallow Clone)
Scenario: The QA team needs to test an update on the gold_core_curated_tbl table. The table is 5TB in size. You cannot afford to duplicate the storage cost just for a test and the update should not affect the copied table.

Task: Create a Shallow Clone of the gold table for the QA team.

Outcome: If we delete records from the source table (gold_core_curated_tbl), will the QA table (gold_core_curated_tbl_qa) be affected? Why or why not?

#### 6. Disaster Recovery (Time Travel & Restore)
Scenario: A junior data engineer accidentally ran a logic error that corrupted the gold_core_curated_tbl table 15 minutes ago. You need to revert the table to its previous state immediately.

Task: Use Delta Lake's Restore feature to roll back the table.

Outcome:What is the difference between querying with VERSION AS OF (Time Travel) and running RESTORE?

In [0]:
%sql
Create catalog If not exists catalog1_we47;
Create Schema If not exists catalog1_we47.Schema_we47;
Create volume If not exists catalog1_we47.Schema_we47.clouddatalake;
Create volume If not exists catalog1_we47.Schema_we47.bronze;


In [0]:
cloudsrc="/Volumes/catalog1_we47/schema_we47/clouddatalake/sourcesystemdata/"#s3 
bronzetgt="/Volumes/catalog1_we47/schema_we47/bronze/ourtargetlocation/"

#Databricks workspace, such as an S3 bucket or a Unity Catalog volume.
ckptlocation="/Volumes/catalog1_we47/schema_we47/clouddatalake/ckpt/_checkpoint"#stores the files copied information post write is successful
schemalocation="/Volumes/catalog1_we47/schema_we47/clouddatalake/_schema"#stores the inferred schema of the source data
df1=spark.readStream.format("cloudFiles")\
.option("cloudFiles.format","csv")\
.option("cloudFiles.maxFilesPerTrigger",1)\
.option("cloudFiles.inferColumnTypes",True)\
.option("cloudFiles.schemaEvolutionMode","addNewColumns")\
.option("checkpointLocation", ckptlocation)\
.option("cloudFiles.schemaLocation", schemalocation)\
.option("header",True)\
.load(cloudsrc)

In [0]:
#realtime trigger is not possible in free serverless
#writeStream will read data from df1 (materialized here) and write to bronzetgt using the schema generated by reader and checkpoint info stored
df1.writeStream.trigger(availableNow=True)\
.option("checkpointLocation", ckptlocation)\
.option("cloudFiles.schemaLocation", schemalocation)\
.option("mergeSchema", "true") \
.outputMode("append")\
.start(bronzetgt)
#.option("mergeSchema", "true") \

In [0]:
# bronzetgt="/Volumes/catalog1_we47/schema_we47/bronze/ourtargetlocation/"
spark.read.format("delta").load(bronzetgt).orderBy("city_name").show(100)

In [0]:
%sql
Select * from catalog1_we47.schema_we47.account

In [0]:
%sql
Select  * from catalog1_we47.schema_we47.account

In [0]:
# 1. Read the CSV from your Data Lake
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/catalog1_we47/schema_we47/sampletest/druginfo.csv")

# 2. Write it as a Delta table
df.write.format("delta").mode("overwrite").saveAsTable("catalog1_we47.schema_we47.druginfotbl")

In [0]:
%sql
Select * from catalog1_we47.schema_we47.druginfotbl

In [0]:
%sql
SELECT * FROM Catalog1_we47.default.drugstbl_medal_gold_mv2

In [0]:
%sql
-- SELECT * FROM Catalog1_we47.default.drugstbl_medal_gold_mv2
SELECT * FROM workspace.default.bronze_staff_data1

In [0]:
%sql
Select * from workspace.default.gold_staff_geo_enriched_dlt2